In [1]:
import os

from PyPDF2 import PdfReader

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

print("All import Successfully...")

All import Successfully...


In [ ]:
ASTRA_DB_APPLICATION_TOKEN = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
ASTRA_DB_ID = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
GOOGLE_API_KEY = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

In [3]:
print(ASTRA_DB_APPLICATION_TOKEN)
print(ASTRA_DB_ID)

AstraCS:DPrihvnmQAmmKUOYLIjKQISl:4d7290ede18c6fe63144d681db5a4bc889cee1c446841b6edc7d1888329ebd2f
6f9af439-0c2e-4e2e-980a-c075ec606c99


In [4]:
pdfreader = PdfReader("National_Budget_speech_2013.pdf")

raw_text = ""
for page in pdfreader.pages:
    text = page.extract_text()
    if text:
        raw_text += text
        
print("Pages: ",len(pdfreader.pages))
print("Characters: ",len(raw_text))

Pages:  39
Characters:  66470


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 200    
)

texts = text_splitter.split_text(raw_text)
print("Chains: ",len(texts))

Chains:  111


In [6]:
import time
import cassio

start = time.time()


# if not ASTRA_DB_APPLICATION_TOKEN or not ASTRA_DB_ID:
#     raise ValueError("Astra DB Token या Database ID missing है")
    
cassio.init(
    token = ASTRA_DB_APPLICATION_TOKEN,
    database_id = ASTRA_DB_ID
)

print("Astra DB Connected Successfully...")
print("Connected in",round(time.time() - start, 2), "seconds")

Astra DB Connected Successfully...
Connected in 11.91 seconds


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001",
    google_api_key = GOOGLE_API_KEY
)
print("Embedding model created Successfully...")

Embedding model created Successfully...


In [8]:
from langchain_community.vectorstores.cassandra import Cassandra
astra_vector_store = Cassandra(
    embedding = embedding,
    table_name = "qa_mini_demo",
    session = None,
    keyspace = None
)
print("Cassandra Vector Store Created Successfully...")

C:\Users\Admin\AppData\Local\Temp\ipykernel_18804\3742245306.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.cassandra import Cassandra


Cassandra Vector Store Created Successfully...


In [9]:
llm = ChatGoogleGenerativeAI(
    model = "models/gemini-3.5-flash",
    temperature = 0.2,
    google_api_key = GOOGLE_API_KEY
)
print("Gemini LLM Loaded Successfully...")

Gemini LLM Loaded Successfully...


In [ ]:
retriever = astra_vector_store.as_retriever(
    search_kwargs={"k" : 5}
)

print("PDF Chatbot Ready!")

while True:
    question = input("\n Ask a question ( type 'quite' to exit?):").strip()
    if question.lower() == "quit":
        print(" Thank you for using the PDF Chatbot!")
        break
    if not question:
        continue
    docs = retriever.invoke(question)
    content = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"Context:\n{content}\n\nQuestion: {question}\nAnswer:"
    response = llm.invoke(prompt)
    print("\nAnswer:", response.content[0]["text"])


PDF Chatbot Ready!



 Ask a question ( type 'quite' to exit?): Global situation 



Answer: As of 2024, the global situation is defined by interconnected challenges and major strategic shifts across several key domains:

1. **Geopolitics and Conflict:**
   * **Major Conflicts:** Ongoing high-intensity conflicts, notably the war in Ukraine and the crisis in the Middle East, continue to drive regional instability and humanitarian emergencies.
   * **Strategic Competition:** Relations among major powers—particularly the US, China, and Russia—remain strained, accelerating a shift toward a more multipolar and fragmented international order.
   * **Expanding Blocs:** Key international groupings are expanding and evolving, as seen in NATO's recent enlargement and the expansion of the BRICS bloc.

2. **Global Economy:**
   * **Inflation and Growth:** Central banks globally have raised interest rates to combat post-pandemic inflation, leading to slower economic growth, though inflation has begun to moderate in many major economies.
   * **Trade and Supply Chains:** Global tra


 Ask a question ( type 'quite' to exit?):  exit



Answer: Goodbye! If you need further assistance later, feel free to ask.
